# Pertemuan 6: Word Embedding dengan Skip-Gram

Notebook ini mendemonstrasikan cara merepresentasikan kata menjadi vektor menggunakan model Skip-Gram (Word2Vec).

**Skip-Gram** adalah model yang memprediksi kata-kata konteks (surrounding words) berdasarkan kata target.

**Mahasiswa:** Wahyu Pratama | **NPM:** 230411100058

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 8)

print("Library berhasil dimuat")

## 2. Definisi 3 Kalimat

Membuat 3 kalimat yang akan digunakan untuk training Skip-Gram model.

In [ ]:
# 3 Kalimat untuk training Skip-Gram
sentences = [
    "saya belajar machine learning untuk text mining",
    "machine learning sangat berguna untuk analisis data",
    "text mining menggunakan teknik machine learning"
]

print("=" * 60)
print("3 KALIMAT UNTUK TRAINING SKIP-GRAM")
print("=" * 60)
for i, sentence in enumerate(sentences, 1):
    print(f"Kalimat {i}: {sentence}")

# Tokenisasi (split menjadi list kata)
tokenized_sentences = [sentence.split() for sentence in sentences]

print("\n" + "=" * 60)
print("TOKENIZED SENTENCES")
print("=" * 60)
for i, tokens in enumerate(tokenized_sentences, 1):
    print(f"Kalimat {i}: {tokens}")

# Ekstrak semua kata unik
all_words = [word for sentence in tokenized_sentences for word in sentence]
unique_words = sorted(set(all_words))

print(f"\nTotal kata: {len(all_words)}")
print(f"Kata unik: {len(unique_words)}")
print(f"Vocabulary: {unique_words}")

## 3. Training Skip-Gram Model

**Skip-Gram** adalah arsitektur Word2Vec yang:
- Input: kata target
- Output: kata-kata konteks di sekitarnya

**Parameter:**
- `vector_size`: dimensi vektor (100)
- `window`: ukuran jendela konteks (2 kata sebelum & sesudah)
- `min_count`: minimum frekuensi kata (1, ambil semua kata)
- `sg=1`: gunakan Skip-Gram (bukan CBOW)
- `epochs`: jumlah iterasi training (100)

In [ ]:
# Training Skip-Gram model
model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,      # Dimensi vektor word embedding
    window=2,             # Context window size
    min_count=1,          # Minimum word frequency
    sg=1,                 # Skip-Gram (1) vs CBOW (0)
    epochs=100,           # Jumlah iterasi training
    seed=42
)

print("=" * 60)
print("SKIP-GRAM MODEL BERHASIL DILATIH")
print("=" * 60)
print(f"Vocabulary size: {len(model.wv)}")
print(f"Vector size: {model.wv.vector_size}")
print(f"Window size: {model.window}")
print(f"Training epochs: {model.epochs}")
print(f"\nVocabulary: {list(model.wv.index_to_key)}")

## 4. Representasi Vektor Setiap Kata

Setiap kata dalam vocabulary direpresentasikan sebagai vektor 100 dimensi.

In [ ]:
print("=" * 60)
print("VEKTOR REPRESENTASI SETIAP KATA")
print("=" * 60)

# Tampilkan vektor untuk setiap kata
for word in unique_words:
    vector = model.wv[word]
    print(f"\nKata: '{word}'")
    print(f"Vektor shape: {vector.shape}")
    print(f"10 dimensi pertama: {vector[:10]}")
    print(f"Norm (magnitude): {np.linalg.norm(vector):.4f}")

## 5. Matriks Vektor Semua Kata

Membuat DataFrame yang berisi vektor representasi untuk semua kata.

In [ ]:
# Buat matriks vektor
word_vectors = {}
for word in unique_words:
    word_vectors[word] = model.wv[word]

# Konversi ke DataFrame
df_vectors = pd.DataFrame(word_vectors).T
df_vectors.columns = [f'dim_{i+1}' for i in range(model.wv.vector_size)]

print("=" * 60)
print("MATRIKS VEKTOR (10 dimensi pertama)")
print("=" * 60)
print(df_vectors.iloc[:, :10])

print(f"\nShape lengkap: {df_vectors.shape}")
print(f"(Jumlah kata: {df_vectors.shape[0]}, Dimensi vektor: {df_vectors.shape[1]})")

## 6. Cosine Similarity antar Kata

Mengukur kemiripan semantik antar kata menggunakan cosine similarity.

**Cosine Similarity:**
- Nilai 1.0: kata identik/sangat mirip
- Nilai 0.0: kata tidak berhubungan
- Nilai -1.0: kata berlawanan makna

In [ ]:
# Hitung cosine similarity
similarity_matrix = cosine_similarity(df_vectors)
df_similarity = pd.DataFrame(
    similarity_matrix,
    index=unique_words,
    columns=unique_words
)

print("=" * 60)
print("COSINE SIMILARITY MATRIX")
print("=" * 60)
print(df_similarity.round(4))

# Visualisasi heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    df_similarity,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    center=0.5,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Similarity'}
)
plt.title('Cosine Similarity Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Kata', fontsize=12)
plt.ylabel('Kata', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Kata-Kata Paling Mirip

Menggunakan fungsi `most_similar()` dari Gensim untuk mencari kata-kata yang paling mirip.

In [ ]:
print("=" * 60)
print("KATA-KATA PALING MIRIP (TOP 3)")
print("=" * 60)

# Pilih beberapa kata kunci untuk analisis
key_words = ['machine', 'learning', 'text', 'mining']

for word in key_words:
    if word in model.wv:
        print(f"\nKata: '{word}'")
        similar_words = model.wv.most_similar(word, topn=3)
        for i, (similar_word, score) in enumerate(similar_words, 1):
            print(f"  {i}. {similar_word} (similarity: {score:.4f})")
    else:
        print(f"\nKata '{word}' tidak ditemukan dalam vocabulary")

## 8. Visualisasi 2D dengan PCA

Mereduksi vektor 100 dimensi menjadi 2 dimensi untuk visualisasi menggunakan PCA.

In [ ]:
# PCA untuk reduksi dimensi ke 2D
pca = PCA(n_components=2, random_state=42)
vectors_2d = pca.fit_transform(df_vectors)

# Buat DataFrame untuk plotting
df_plot = pd.DataFrame({
    'word': unique_words,
    'x': vectors_2d[:, 0],
    'y': vectors_2d[:, 1]
})

print("=" * 60)
print("KOORDINAT 2D SETELAH PCA")
print("=" * 60)
print(df_plot)

# Visualisasi scatter plot
plt.figure(figsize=(12, 8))

# Plot points
plt.scatter(df_plot['x'], df_plot['y'], s=200, alpha=0.6, c='steelblue', edgecolors='black', linewidth=1.5)

# Annotate each point dengan nama kata
for idx, row in df_plot.iterrows():
    plt.annotate(
        row['word'],
        (row['x'], row['y']),
        fontsize=12,
        fontweight='bold',
        ha='center',
        va='bottom',
        xytext=(0, 5),
        textcoords='offset points'
    )

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('Visualisasi 2D Word Embeddings (Skip-Gram)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3, linestyle='--')
plt.axhline(y=0, color='gray', linewidth=0.8, alpha=0.5)
plt.axvline(x=0, color='gray', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

print(f"\nVariance explained by 2 components: {pca.explained_variance_ratio_.sum():.2%}")

## 9. Operasi Aritmatika Vektor

Word embeddings memungkinkan operasi matematika seperti:
- King - Man + Woman ≈ Queen
- Paris - France + Italy ≈ Rome

Kita coba dengan kata-kata dalam vocabulary kita.

In [ ]:
print("=" * 60)
print("OPERASI ARITMATIKA VEKTOR")
print("=" * 60)

# Contoh: machine + text
if 'machine' in model.wv and 'text' in model.wv:
    result_vector = model.wv['machine'] + model.wv['text']
    print("\nOperasi: machine + text")
    print(f"Hasil vektor shape: {result_vector.shape}")
    print(f"10 dimensi pertama: {result_vector[:10]}")
    
    # Cari kata terdekat dengan hasil vektor
    similar = model.wv.similar_by_vector(result_vector, topn=3)
    print("\nKata paling mirip dengan hasil 'machine + text':")
    for word, score in similar:
        print(f"  - {word} (similarity: {score:.4f})")

# Contoh: learning - text
if 'learning' in model.wv and 'text' in model.wv:
    result_vector = model.wv['learning'] - model.wv['text']
    print("\n" + "-" * 60)
    print("Operasi: learning - text")
    print(f"Hasil vektor shape: {result_vector.shape}")
    print(f"10 dimensi pertama: {result_vector[:10]}")
    
    similar = model.wv.similar_by_vector(result_vector, topn=3)
    print("\nKata paling mirip dengan hasil 'learning - text':")
    for word, score in similar:
        print(f"  - {word} (similarity: {score:.4f})")

## 10. Analisis Context Window

Menunjukkan bagaimana Skip-Gram menggunakan context window untuk belajar.

In [ ]:
print("=" * 60)
print("CONTOH CONTEXT WINDOW (window=2)")
print("=" * 60)

# Ambil kalimat pertama sebagai contoh
example_sentence = tokenized_sentences[0]
print(f"\nKalimat: {' '.join(example_sentence)}")
print("\nSkip-Gram training pairs (target -> context):")

window_size = 2
for i, target in enumerate(example_sentence):
    # Tentukan context words (window size = 2)
    start = max(0, i - window_size)
    end = min(len(example_sentence), i + window_size + 1)
    context = [example_sentence[j] for j in range(start, end) if j != i]
    
    print(f"\nTarget: '{target}'")
    print(f"Context words: {context}")
    print(f"Training pairs:")
    for ctx_word in context:
        print(f"  '{target}' -> '{ctx_word}'")

## 11. Ringkasan & Kesimpulan

**Skip-Gram Word2Vec** berhasil merepresentasikan setiap kata menjadi vektor numerik yang menangkap makna semantik kata.

**Keunggulan:**
- Kata dengan makna mirip memiliki vektor yang mirip (cosine similarity tinggi)
- Mendukung operasi aritmatika vektor
- Dapat divisualisasikan dalam 2D/3D

**Hasil:**
- 3 kalimat digunakan untuk training
- Setiap kata direpresentasikan sebagai vektor 100 dimensi
- Kata-kata seperti 'machine' dan 'learning' memiliki similarity tinggi
- Context window size = 2 menangkap hubungan lokal antar kata

In [ ]:
print("=" * 60)
print("RINGKASAN SKIP-GRAM MODEL")
print("=" * 60)

summary = {
    'Jumlah Kalimat': len(sentences),
    'Total Kata': len(all_words),
    'Vocabulary Size': len(unique_words),
    'Vector Dimensi': model.wv.vector_size,
    'Window Size': model.window,
    'Training Epochs': model.epochs,
    'Model Type': 'Skip-Gram'
}

for key, value in summary.items():
    print(f"{key:.<40} {value}")

print("\n" + "=" * 60)
print("Mahasiswa: Wahyu Pratama | NPM: 230411100058")
print("=" * 60)